# AI vs Real Face Detector — Run 2 Dataset Preparation
## FFHQ Real + StyleGAN2 + Diffusion DeepFakeFace

This is the consolidated Run 2 dataset-preparation notebook.

It combines:
- the verified `OpenRL/DeepFakeFace` diffusion source and ZIP-only download,
- the original Kaggle FFHQ and fake-face download workflow,
- automatic recreation of the deleted `source/ffhq` and `source/stylegan2` folders,
- balanced 5,000-real / 5,000-fake sampling,
- source-stratified 70/15/15 train/validation/test splits,
- final verification and reproducibility manifests.

**Important:** the Kaggle `hyperclaw79/fakefaces` dataset is used exactly as in the original uploaded notebook for the StyleGAN2 source. Verify its contents before training if generator provenance is critical.


In [ ]:
# 1. Configuration
from pathlib import Path
import os, random, shutil, zipfile, json, glob

SEED = 42
random.seed(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/ai-vs-real-face-detector")
SOURCE_DIR = DRIVE_ROOT / "source"
DATA_DIR = DRIVE_ROOT / "data"

FFHQ_SOURCE = SOURCE_DIR / "ffhq"
STYLEGAN_SOURCE = SOURCE_DIR / "stylegan2"

N_REAL = 5000
N_STYLEGAN = 2500
N_DIFFUSION_TEXT2IMG = 1250
N_DIFFUSION_INPAINTING = 1250

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

for p in [FFHQ_SOURCE, STYLEGAN_SOURCE]:
    p.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        (DATA_DIR / split / label).mkdir(parents=True, exist_ok=True)

print("Project:", DRIVE_ROOT)
print("FFHQ source:", FFHQ_SOURCE)
print("StyleGAN2 source:", STYLEGAN_SOURCE)
print("DATA_DIR:", DATA_DIR)


In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Install download dependencies
!pip -q install -U kaggle huggingface_hub


In [ ]:
# 4. Configure Kaggle authentication
from google.colab import files
import os

KAGGLE_PATH = Path("/root/.kaggle/kaggle.json")
KAGGLE_PATH.parent.mkdir(parents=True, exist_ok=True)

if not KAGGLE_PATH.exists():
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No kaggle.json was uploaded.")
    # Accept the uploaded file regardless of its original filename.
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, KAGGLE_PATH)

os.chmod(KAGGLE_PATH, 0o600)
print("Kaggle credentials configured.")


In [ ]:
# 5. Recreate/download FFHQ source if the deleted source folder is empty.
# This follows the FFHQ download source from the original dataset-prep notebook.

N_REAL_SOURCE = 5000
FFHQ_DL = Path("/content/ffhq_dl")

def image_files(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted([
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    ])

existing_ffhq = image_files(FFHQ_SOURCE)
print("Existing FFHQ source images:", len(existing_ffhq))

if len(existing_ffhq) < N_REAL_SOURCE:
    if not (FFHQ_DL / "download_complete.txt").exists():
        !rm -rf /content/ffhq_dl
        !mkdir -p /content/ffhq_dl
        !kaggle datasets download -d pankymathur/ffhq-224k -p /content/ffhq_dl --unzip
        (FFHQ_DL / "download_complete.txt").touch()

    ffhq_candidates = image_files(FFHQ_DL)
    print("Downloaded FFHQ candidates:", len(ffhq_candidates))
    assert len(ffhq_candidates) >= N_REAL_SOURCE, (
        f"Need at least {N_REAL_SOURCE} FFHQ images; found {len(ffhq_candidates)}."
    )

    random.seed(SEED)
    random.shuffle(ffhq_candidates)

    # Copy only enough images to rebuild the source folder.
    for idx, src in enumerate(ffhq_candidates[:N_REAL_SOURCE]):
        dst = FFHQ_SOURCE / f"ffhq_{idx:06d}{src.suffix.lower()}"
        if not dst.exists():
            shutil.copy2(src, dst)

existing_ffhq = image_files(FFHQ_SOURCE)
print("FFHQ source ready:", len(existing_ffhq))
assert len(existing_ffhq) >= N_REAL


In [ ]:
# 6. Recreate/download StyleGAN2 source if the deleted source folder is empty.
# This follows the StyleGAN/fake-face source from the original dataset-prep notebook.

N_STYLEGAN_SOURCE = 2500
STYLEGAN_DL = Path("/content/fake_dl")

existing_stylegan = image_files(STYLEGAN_SOURCE)
print("Existing StyleGAN2 source images:", len(existing_stylegan))

if len(existing_stylegan) < N_STYLEGAN_SOURCE:
    if not (STYLEGAN_DL / "download_complete.txt").exists():
        !rm -rf /content/fake_dl
        !mkdir -p /content/fake_dl
        !kaggle datasets download -d hyperclaw79/fakefaces -p /content/fake_dl --unzip
        (STYLEGAN_DL / "download_complete.txt").touch()

    stylegan_candidates = image_files(STYLEGAN_DL)
    print("Downloaded fake-face candidates:", len(stylegan_candidates))
    assert len(stylegan_candidates) >= N_STYLEGAN_SOURCE, (
        f"Need at least {N_STYLEGAN_SOURCE} fake-face images; found {len(stylegan_candidates)}."
    )

    random.seed(SEED)
    random.shuffle(stylegan_candidates)

    for idx, src in enumerate(stylegan_candidates[:N_STYLEGAN_SOURCE]):
        dst = STYLEGAN_SOURCE / f"stylegan2_{idx:06d}{src.suffix.lower()}"
        if not dst.exists():
            shutil.copy2(src, dst)

existing_stylegan = image_files(STYLEGAN_SOURCE)
print("StyleGAN2 source ready:", len(existing_stylegan))
assert len(existing_stylegan) >= N_STYLEGAN


In [ ]:
# 7. Verify the public diffusion dataset before downloading
from huggingface_hub import HfApi

REPO_ID = "OpenRL/DeepFakeFace"
api = HfApi()
info = api.dataset_info(REPO_ID)

print("Dataset:", info.id)
print("License:", getattr(info, "license", None))
print("Public dataset metadata loaded successfully.")


In [ ]:
# 8. Download ONLY the two diffusion ZIP files we need.
# This intentionally uses OpenRL/DeepFakeFace, not the old Purdue-M2 repo.

from huggingface_hub import snapshot_download

HF_CACHE = "/content/deepfakeface_hf"

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=HF_CACHE,
    allow_patterns=["text2img.zip", "inpainting.zip"],
)

TEXT2IMG_ZIP = Path(HF_CACHE) / "text2img.zip"
INPAINT_ZIP = Path(HF_CACHE) / "inpainting.zip"

print("text2img:", TEXT2IMG_ZIP.exists(), TEXT2IMG_ZIP)
print("inpainting:", INPAINT_ZIP.exists(), INPAINT_ZIP)

assert TEXT2IMG_ZIP.exists()
assert INPAINT_ZIP.exists()


In [ ]:
# 9. Extract only the required number of diffusion images from each ZIP.

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

EXTRACT_ROOT = Path("/content/deepfakeface_selected")
TEXT2IMG_DIR = EXTRACT_ROOT / "text2img"
INPAINT_DIR = EXTRACT_ROOT / "inpainting"

TEXT2IMG_DIR.mkdir(parents=True, exist_ok=True)
INPAINT_DIR.mkdir(parents=True, exist_ok=True)

def extract_n_images(zip_path, out_dir, n, seed=42):
    existing = image_files(out_dir)
    if len(existing) >= n:
        return existing[:n]

    with zipfile.ZipFile(zip_path, "r") as z:
        members = [
            m for m in z.namelist()
            if not m.endswith("/") and Path(m).suffix.lower() in IMAGE_EXTS
        ]
        if len(members) < n:
            raise RuntimeError(f"{zip_path} has only {len(members)} images; need {n}")

        rng = random.Random(seed)
        selected = rng.sample(members, n)

        for idx, member in enumerate(selected):
            ext = Path(member).suffix.lower()
            out = out_dir / f"{idx:06d}{ext}"
            if out.exists():
                continue
            with z.open(member) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)

    return image_files(out_dir)

text2img_files = extract_n_images(TEXT2IMG_ZIP, TEXT2IMG_DIR, N_DIFFUSION_TEXT2IMG, SEED)
inpaint_files = extract_n_images(INPAINT_ZIP, INPAINT_DIR, N_DIFFUSION_INPAINTING, SEED + 1)

print("Selected SD text2img:", len(text2img_files))
print("Selected SD inpainting:", len(inpaint_files))

assert len(text2img_files) >= N_DIFFUSION_TEXT2IMG
assert len(inpaint_files) >= N_DIFFUSION_INPAINTING


In [ ]:
# 10. Build balanced source pools.
random.seed(SEED)

real_files = image_files(FFHQ_SOURCE)
stylegan_files = image_files(STYLEGAN_SOURCE)

real_selected = random.sample(real_files, N_REAL)
stylegan_selected = random.sample(stylegan_files, N_STYLEGAN)

fake_selected = (
    [(p, "stylegan2") for p in stylegan_selected] +
    [(p, "sd_text2img") for p in text2img_files[:N_DIFFUSION_TEXT2IMG]] +
    [(p, "sd_inpainting") for p in inpaint_files[:N_DIFFUSION_INPAINTING]]
)

assert len(real_selected) == N_REAL
assert len(fake_selected) == 5000

print("Real:", len(real_selected))
print("Fake:", len(fake_selected))


In [ ]:
# 11. Split each class while preserving generator/source identity.
def split_items(items, seed=42):
    items = list(items)
    rng = random.Random(seed)
    rng.shuffle(items)

    n = len(items)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    return {
        "train": items[:n_train],
        "val": items[n_train:n_train+n_val],
        "test": items[n_train+n_val:]
    }

real_splits = split_items([(p, "ffhq") for p in real_selected], SEED)
fake_splits = split_items(fake_selected, SEED + 100)

for split in ["train", "val", "test"]:
    print(split, "real:", len(real_splits[split]), "fake:", len(fake_splits[split]))


In [ ]:
# 12. Copy images into the exact structure expected by full_hybrid training.
# Clear previous split folders first so reruns cannot leave stale files.

for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        dest = DATA_DIR / split / label
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)

def safe_copy(items, split, label):
    dest = DATA_DIR / split / label
    for idx, (src, source_name) in enumerate(items):
        src = Path(src)
        out = dest / f"{source_name}_{idx:06d}{src.suffix.lower()}"
        shutil.copy2(src, out)

for split in ["train", "val", "test"]:
    safe_copy(real_splits[split], split, "real")
    safe_copy(fake_splits[split], split, "fake")

print("Dataset copied to:", DATA_DIR)


In [ ]:
# 13. Final verification
def count_images(path):
    return sum(
        1 for p in Path(path).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

counts = {}
for split in ["train", "val", "test"]:
    counts[split] = {}
    for label in ["real", "fake"]:
        counts[split][label] = count_images(DATA_DIR / split / label)

print(json.dumps(counts, indent=2))

expected = {
    "train": {"real": 3500, "fake": 3500},
    "val": {"real": 750, "fake": 750},
    "test": {"real": 750, "fake": 750},
}
assert counts == expected, f"Unexpected split counts: {counts}"

print("Dataset structure and counts are correct.")


In [ ]:
# 14. Preview a few files
from PIL import Image
import matplotlib.pyplot as plt

samples = []
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        files_ = image_files(DATA_DIR / split / label)
        samples.extend([(split, label, p) for p in files_[:2]])

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for ax, (split, label, path) in zip(axes.ravel(), samples[:12]):
    try:
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(f"{split}/{label}")
        ax.axis("off")
    except Exception as e:
        ax.set_title(f"ERROR: {e}")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 15. Save reproducibility manifests
import pandas as pd

rows = []
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        for p in image_files(DATA_DIR / split / label):
            rows.append({
                "split": split,
                "label": label,
                "path": str(p)
            })

manifest_df = pd.DataFrame(rows)
manifest_dir = DATA_DIR / "splits"
manifest_dir.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    manifest_df[manifest_df["split"] == split].to_csv(
        manifest_dir / f"{split}.csv", index=False
    )

manifest = {
    "seed": SEED,
    "real": {"source": "FFHQ", "count": N_REAL},
    "fake": {
        "stylegan2": N_STYLEGAN,
        "stable_diffusion_v1_5_text2img": N_DIFFUSION_TEXT2IMG,
        "stable_diffusion_inpainting": N_DIFFUSION_INPAINTING,
    },
    "split": {
        "train": TRAIN_RATIO,
        "val": VAL_RATIO,
        "test": 1 - TRAIN_RATIO - VAL_RATIO,
    },
    "data_dir": str(DATA_DIR),
    "source_dirs": {
        "ffhq": str(FFHQ_SOURCE),
        "stylegan2": str(STYLEGAN_SOURCE),
    },
}

manifest_path = DATA_DIR / "run2_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print("Saved:", manifest_path)
print("Saved CSVs:", list(manifest_dir.glob("*.csv")))


# Done

The Run 2 dataset is now:

```text
ai-vs-real-face-detector/
├── source/
│   ├── ffhq/
│   └── stylegan2/
└── data/
    ├── train/
    │   ├── real/
    │   └── fake/
    ├── val/
    │   ├── real/
    │   └── fake/
    ├── test/
    │   ├── real/
    │   └── fake/
    ├── splits/
    └── run2_manifest.json
```

Expected counts:

```text
train/real: 3500
train/fake: 3500
val/real:     750
val/fake:     750
test/real:    750
test/fake:    750
```

Fake composition:
- 1,750 StyleGAN2
- 875 Stable Diffusion v1.5 text-to-image
- 875 Stable Diffusion Inpainting

Run this notebook before the training notebook if the deleted source folders need to be rebuilt.
